## Step 1: Import Libraries

We start by importing pandas for loading and manipulating data, and a few utilities from scikit-learn for preprocessing and modelling. These are all standard libraries that come pre-installed in most Python environments like Kaggle, Colab, or Anaconda. No additional installations are required to run this notebook.


In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## Step 2: Exploratory Data Analysis
Loading datasets and understanding the data structure, 
missing values, and target distribution.
We read all three CSV files — train.csv, public_test.csv, and private_test.csv — into pandas DataFrames. The training set contains labelled examples we will learn from, while the private test set is what we need to make final predictions on. Printing the shapes gives a quick sanity check that the files loaded correctly.

In [2]:
train_raw= pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")
train =train_raw.copy()
print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

train.head(10)


Train Shape: (10000, 14)
Public Test Shape: (3000, 14)
Private Test Shape: (3000, 13)


,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0
5,6,40.0,NaN,2,Mobile,Paid Ads,27,30,17.10,1,0,16,8638,0
6,7,21.0,64908.075218,2,Mobile,Email,30,26,16.59,1,0,11,7266,0
7,8,35.0,114200.619715,1,Desktop,Paid Ads,7,7,2.35,1,1,23,6252,1
8,9,48.0,57021.611431,3,Mobile,Paid Ads,6,9,14.49,6,1,20,7480,1
9,10,35.0,66561.117857,1,Desktop,Organic,1,7,160.24,6,0,20,7906,1


### Step 3: Define Features and Target

We separate the target column (Converted) from the rest of the features in the training set. For simplicity, we only keep numeric columns in this baseline — categorical columns are dropped for now. This keeps things easy to understand before we add more advanced preprocessing.


In [3]:
TARGET = "Converted"
##In give starter notebook i think device type and traffic source are dropped but it is important and i will encode it
# Separate features and target
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_public = public_test.drop(columns=[TARGET])
y_public = public_test[TARGET]

X_private=private_test.copy()   #no TARGET in private test so only features
# Keep only numeric columns for this baseline
#numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
#X_train = X_train[numeric_cols]
#X_private = private_test[numeric_cols]
#X_public = public_test[numeric_cols]
#since userid is unique and of no use so dropping them
X_train = X_train.drop(columns=['User_ID'])
X_public = X_public.drop(columns=['User_ID'])
X_private = X_private.drop(columns=['User_ID'])
#print("Features used:", numeric_cols)
X_train


,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code
0,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418
1,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213
2,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849
3,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610
4,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,65.0,90204.794616,1,Mobile,Organic,9,9,4.67,5,0,6,8293
9996,53.0,93283.066383,1,Mobile,Organic,22,21,NaN,3,1,23,9175
9997,49.0,75347.420328,2,Desktop,Paid Ads,19,19,5.18,1,1,4,9709
9998,51.0,59311.028526,2,Mobile,Organic,15,13,5.67,1,1,1,2449


### Step 4 : Checking the Conversion rate, Null values and Unique Values

In [4]:
print(train['Converted'].value_counts())
#I am calculating the proportion
print(train['Converted'].value_counts(normalize=True) * 100)

Converted
0    6913
1    3087
Name: count, dtype: int64
Converted
0    69.13
1    30.87
Name: proportion, dtype: float64


In [5]:
print(train.isnull().sum())

User_ID                  0
Age                   1480
Income                 984
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          1848
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64


In [6]:
print(train['Device_Type'].unique())
print(train['Traffic_Source'].unique())
X_private

<ArrowStringArray>
['Mobile', 'Desktop', 'Tablet']
Length: 3, dtype: str
<ArrowStringArray>
['Organic', 'Social Media', 'Referral', 'Email', 'Paid Ads']
Length: 5, dtype: str


,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code
0,54.0,12000.000000,2,Mobile,Email,3,1,13.14,3,0,9,3685
1,59.0,NaN,3,Mobile,Organic,22,20,NaN,4,0,18,9612
2,NaN,79296.908337,2,Mobile,Paid Ads,28,24,3.36,1,0,2,4522
3,19.0,65279.025663,2,Mobile,Social Media,20,13,8.78,6,1,5,4623
4,31.0,NaN,3,Mobile,Paid Ads,10,15,24.14,5,0,20,8663
...,...,...,...,...,...,...,...,...,...,...,...,...
2995,32.0,75294.529879,2,Mobile,Paid Ads,20,18,10.68,4,0,12,1794
2996,30.0,86278.626308,3,Tablet,Organic,8,8,15.75,4,1,11,4526
2997,53.0,76122.507493,2,Desktop,Organic,28,30,7.93,1,1,24,9217
2998,29.0,78503.583072,2,Mobile,Social Media,25,30,20.11,2,1,17,7176


## Step 5: Encoding Categorical Features

Device_Type and Traffic_Source are text columns that my model cannot 
understand directly. I am converting them into binary (0/1) columns 
using one-hot encoding. For example, Device_Type becomes three separate 
columns: Device_Type_Mobile, Device_Type_Desktop, Device_Type_Tablet.
I apply this to all three datasets consistently.

In [7]:
# i want to convert device type and traffic source in binary because my model needs to understand it.

X_train = pd.get_dummies(X_train, columns=['Device_Type', 'Traffic_Source'],dtype= int)
X_public = pd.get_dummies(X_public, columns=['Device_Type', 'Traffic_Source'], dtype=int)
X_private = pd.get_dummies(X_private, columns=['Device_Type', 'Traffic_Source'], dtype=int)
X_train

,Age,Income,City_Tier,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Device_Type_Desktop,Device_Type_Mobile,Device_Type_Tablet,Traffic_Source_Email,Traffic_Source_Organic,Traffic_Source_Paid Ads,Traffic_Source_Referral,Traffic_Source_Social Media
0,58.0,103593.708812,2,5,4,9.61,3,0,11,2418,0,1,0,0,1,0,0,0
1,26.0,36451.716984,2,11,3,17.63,2,0,14,1213,0,1,0,0,0,0,0,1
2,19.0,30511.228700,3,1,1,13.25,5,0,5,2849,0,1,0,0,0,0,1,0
3,48.0,87789.172342,3,14,12,NaN,1,1,19,7610,0,1,0,1,0,0,0,0
4,35.0,105229.249067,2,14,21,16.92,1,0,5,9261,0,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,65.0,90204.794616,1,9,9,4.67,5,0,6,8293,0,1,0,0,1,0,0,0
9996,53.0,93283.066383,1,22,21,NaN,3,1,23,9175,0,1,0,0,1,0,0,0
9997,49.0,75347.420328,2,19,19,5.18,1,1,4,9709,1,0,0,0,0,1,0,0
9998,51.0,59311.028526,2,15,13,5.67,1,1,1,2449,0,1,0,0,1,0,0,0


## Step 6: Aligning Columns Across Datasets

After one-hot encoding, train, public, and private datasets might have 
different columns if a category appears in one dataset but not another.
I use reindex() to ensure all three datasets have identical columns.
Missing categories are filled with 0.

In [8]:
# Align public and private to have same columns as train if needed then reindex
#and if not present then fill the values with 0
#this is to prevent my data from mismatching the columns
X_public = X_public.reindex(columns=X_train.columns, fill_value=0)
X_private = X_private.reindex(columns=X_train.columns, fill_value=0)
X_public.isnull().sum()

Age                            472
Income                         301
City_Tier                        0
Pages_Viewed                     0
Products_Viewed                  0
Time_On_Site                   568
Previous_Purchases               0
Discount_Seen                    0
Browser_Version                  0
Campaign_Code                    0
Device_Type_Desktop              0
Device_Type_Mobile               0
Device_Type_Tablet               0
Traffic_Source_Email             0
Traffic_Source_Organic           0
Traffic_Source_Paid Ads          0
Traffic_Source_Referral          0
Traffic_Source_Social Media      0
dtype: int64

## Step 7: Featuring the critical datapoints
Creating new features to capture user behavior patterns.

In [9]:
X_train['page_per_time']=X_train['Pages_Viewed']/(X_train['Time_On_Site']+0.001)
X_public['page_per_time']=X_public['Pages_Viewed']/(X_public['Time_On_Site']+0.001)
X_private['page_per_time']=X_private['Pages_Viewed']/(X_private['Time_On_Site']+0.001)

X_train['products_per_page']=X_train['Products_Viewed']/(X_train['Pages_Viewed']+0.001)
X_public['products_per_page']=X_public['Products_Viewed']/(X_public['Pages_Viewed']+0.001)
X_private['products_per_page']=X_private['Products_Viewed']/(X_private['Pages_Viewed']+0.001)
# Creating Age Groups for  teen,young adult,aduit,senior to cut them in categories and the model should treat them different 
age_bins = [0, 25, 35, 50, 100]
age_labels = [1, 2, 3, 4]

X_train['Age_Group'] = pd.cut(X_train['Age'], bins=age_bins, labels=age_labels)
X_public['Age_Group'] = pd.cut(X_public['Age'], bins=age_bins, labels=age_labels)
X_private['Age_Group'] = pd.cut(X_private['Age'], bins=age_bins, labels=age_labels)


import numpy as np

# Create Hot Lead Flag (1 if true, 0 if false)
X_train['Hot_Lead'] = np.where((X_train['Previous_Purchases'] > 0) & (X_train['Discount_Seen'] == 1), 1, 0)
X_public['Hot_Lead'] = np.where((X_public['Previous_Purchases'] > 0) & (X_public['Discount_Seen'] == 1), 1, 0)
X_private['Hot_Lead'] = np.where((X_private['Previous_Purchases'] > 0) & (X_private['Discount_Seen'] == 1), 1, 0)




## Step 8: Handling Missing Values and Feature Scaling

Three columns have missing values: Age (14.8%), Income (9.84%), 
Time_On_Site (18.5%). I fill them using median imputation.

Important: I fit the imputer and scaler ONLY on training data, 
then apply the same transformation to public and private test data.
This prevents data leakage — the model should not learn any 
information from test data during training.

StandardScaler brings all features to the same scale (zero mean, 
unit variance) which improves Logistic Regression performance.

In [10]:
# Now I need to fill NaN values with median etc..
# Fill missing values with the median of each column## Step 5: Handling Missing Values and Feature Scaling

#Basically I want to do fit transform on train data only 
#and I dont want to relearn from public and private data so only transform
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_private_imputed = imputer.transform(X_private)
X_public_imputed = imputer.transform(X_public)

# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_private_scaled = scaler.transform(X_private_imputed)
X_public_scaled = scaler.transform(X_public_imputed)
print("Preprocessing done! Training data shape:", X_train_scaled.shape)

Preprocessing done! Training data shape: (10000, 22)


 ## Step 9: Model Training and Comparison

I trained three classification models and compared F1 scores on 
public_test.csv. F1 score is used instead of accuracy because the 
target is imbalanced (69% non-converted, 31% converted).

class_weight='balanced' is used in Logistic Regression and Random 
Forest to ensure the model pays attention to the minority class 
(converted users) and does not simply predict 0 for everyone.

In [11]:
!pip install xgboost

In [12]:
from sklearn.metrics import f1_score


lrmodel = LogisticRegression(max_iter=1000, random_state=42,class_weight='balanced') 
#random state is fixed so that we can the same.
#weight class balanced because my 0's and 1's are not 50-50 so i need to take care of missing out the minority
lrmodel.fit(X_train_scaled, y_train)

print("Model trained successfully!")
#print("Training Accuracy: {:.4f}".format(model.score(X_train_scaled, y_train)))
#training accuracy was 0.6750
y_public_pred = lrmodel.predict(X_public_scaled)
print("Baseline F1 Score:", f1_score(y_public, y_public_pred))
#so my baseline model is done and it has given F1 score of 0.5422503389064618


Model trained successfully!
Baseline F1 Score: 0.5438121047877146


In [13]:
# now to improve my f1 socre i am mivng to a tree like model 

from sklearn.ensemble import RandomForestClassifier 
rfmodel= RandomForestClassifier(n_estimators=100,
                                random_state =33,class_weight='balanced',
                                max_depth=7,min_samples_split=10) 
rfmodel.fit(X_train_scaled,y_train)
print("Model trained succesfully")
y_public_predrf = rfmodel.predict(X_public_scaled)
print("RF model F1 score:" ,f1_score(y_public,y_public_predrf))


Model trained succesfully
RF model F1 score: 0.5251798561151079


In [14]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    scale_pos_weight=2.2, 
    max_depth=5, 
    learning_rate=0.1, 
    random_state=42
)
xgb_model.fit(X_train_scaled,y_train)
print("Model trained succesfully")
y_public_predxgb= xgb_model.predict(X_public_scaled)
print("xgb model F1 score:" ,f1_score(y_public,y_public_predxgb))


Model trained succesfully
xgb model F1 score: 0.5380009212344542


## Step 10: Final Model Selection and Submission

After comparing all three models:
- Logistic Regression: F1 = 0.5438 (best)
- XGBoost: F1 = 0.5380  
- Random Forest: F1 = 0.5252

Logistic Regression performed best, suggesting a roughly linear 
relationship between features and conversion probability.
Final predictions are generated using Logistic Regression on 
private_test.csv and saved as submission.csv.

In [15]:
predictions = lrmodel.predict(X_private_scaled)

submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
submission.head()

submission.csv created successfully!


,User_ID,Converted
0,103001,0
1,103002,0
2,103003,0
3,103004,1
4,103005,0
